# 📖 Notebook 8: Temporal for AI Agents — Durable, Observable, Human-Supervised AI

AI agents look magical in demos, but in real systems they are **long-running**, **failure-prone**, and often need **human approval** before they take an action. A normal Python loop loses its place when the process crashes. A Temporal workflow keeps the agent's state safe, replays from event history, and lets outside systems inspect or guide the agent while it is still running.

Think of Temporal as the agent's durable memory + control tower:
- the **workflow** stores the conversation and the plan
- **activities** call external systems like LLMs or tools
- **signals** let humans approve or reject risky actions
- **queries** let dashboards ask, "What is the agent doing right now?"

## Learning Objectives

- Understand why AI agents need durable execution
- Wrap LLM calls as Activities with retry policies
- Persist conversation state in Workflow variables
- Add human-in-the-loop approval using Signals
- Query agent status from external systems
- Build a tool-calling loop as a durable Workflow
- Use Continue-As-New for long-lived agent sessions


## 🛠️ Setup

Make sure Temporal and the Temporal UI are running:

```bash
cd 03-technologies/workflow-engines/temporal
uv sync
docker compose up -d
```

If this is your first time opening these notebooks, register the kernel once:

```bash
uv run python -m ipykernel install --user --name temporal --display-name "Temporal (Python)"
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import asyncio
import uuid
from dataclasses import dataclass, field
from datetime import timedelta

from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.common import RetryPolicy
from temporalio.worker import Worker

client = await Client.connect("localhost:7233")
TASK_QUEUE = "ai-agent-task-queue"
print("✅ Connected to Temporal")


---
## 🤔 Why Temporal for AI Agents?

AI agents fail in all the same ways as other distributed systems — and then add extra problems like expensive retries, partial tool execution, and human approvals that may take minutes or hours.

```
Without Temporal:                With Temporal:
─────────────────               ──────────────
LLM API fails → crash           LLM API fails → automatic retry
Server restarts → lost          Server restarts → resume from last step
No visibility into agent        Query agent status anytime
No approval flow                Signal to approve/reject actions
Memory grows forever            Continue-As-New to compact
```

A good mental model is: **an AI agent is not a single request — it is a long-lived conversation with state, tools, retries, and supervision**. Temporal is built for exactly that kind of work.


## 🏗️ Architecture

```
┌─────────────┐      ┌──────────────┐      ┌───────────────┐
│  User / API │─────▶│  Agent       │─────▶│  Activities    │
│             │      │  Workflow    │      │               │
│  send msg   │      │  (durable    │      │  call_llm()   │
│  approve    │      │   state)     │      │  search_web() │
│  query      │      │              │      │  execute_code()│
└─────────────┘      └──────┬───────┘      └───────────────┘
                             │
                    ┌────────▼────────┐
                    │  Temporal Server │
                    │  (event history) │
                    └─────────────────┘
```

The user or API never talks to the tools directly. Instead, it talks to the **workflow execution**. The workflow decides what to do next, stores the current state, and asks Activities to perform side effects.


---
## 🤖 Part 1: LLM calls as Activities

An LLM call touches the outside world: network, vendor API, latency, rate limits, and transient errors. That means it belongs in an **Activity**, not inside the Workflow code itself.

Why this matters:
- Activities can retry automatically.
- Activities can time out.
- Activities can log, call APIs, and handle failures.
- Workflow code stays deterministic and replay-safe.

For this lab we simulate the LLM so you need **no API key**.


In [ ]:
@dataclass
class LlmRequest:
    prompt: str
    system_message: str = "You are a helpful assistant."
    max_tokens: int = 500


@dataclass
class LlmResponse:
    text: str
    model: str = "simulated-gpt"
    tokens_used: int = 0


@activity.defn
async def call_llm(request: LlmRequest) -> LlmResponse:
    """In production, this calls OpenAI, Azure OpenAI, or Anthropic."""
    activity.logger.info(f"LLM call: {request.prompt[:80]}...")
    await asyncio.sleep(0.5)

    prompt_lower = request.prompt.lower()
    if "summarize" in prompt_lower:
        text = "Summary: The document explains how Temporal keeps long-running work durable and observable."
    elif "analyze" in prompt_lower:
        text = "Analysis: The agent should separate durable state from external side effects and retry transient failures."
    elif "plan" in prompt_lower:
        text = "Plan: 1) Start with one workflow per agent session, 2) wrap tool calls as activities, 3) add approval signals, 4) monitor with queries."
    elif "tool outputs" in prompt_lower:
        text = "Final answer: Temporal makes the agent durable, observable, and easier to supervise. Start with a small pilot, keep tool calls in activities, and add approval gates around risky actions."
    else:
        text = f"I processed your request: '{request.prompt[:50]}...'. Here's my response."

    return LlmResponse(text=text, tokens_used=len(text.split()) * 2)


print("✅ Defined call_llm() as a retryable Activity")


---
## 🧰 Part 2: Tools are Activities too

Tools are just side effects with a friendly name. A web search, a file read, a database lookup, or code execution all belong in Activities because they may fail and need retries.

In a real AI agent, the LLM would usually decide which tool to call. In this notebook we use a simple keyword-based planner so the flow stays beginner-friendly.


In [ ]:
@activity.defn
async def search_web(query: str) -> str:
    """Simulated web search tool."""
    await asyncio.sleep(0.3)
    return f"Search results for '{query}': [Result 1: Temporal docs, Result 2: AI agent architecture article]"


@activity.defn
async def execute_code(code: str) -> str:
    """Simulated code execution tool."""
    await asyncio.sleep(0.2)
    if "6 * 7" in code or "6*7" in code:
        return "Code executed successfully. Output: 42"
    return "Code executed successfully. Output: simulated result"


print("✅ Defined tool Activities: search_web(), execute_code()")


---
## 🧠 Part 3: The agent itself is a durable Workflow

The Workflow is the agent's brain that never forgets.

It keeps these values in Workflow state:
- conversation history
- current status
- total tokens used
- tools used so far
- pending human decisions

This is the key idea: **workflow variables are durable**. If the worker process crashes, Temporal replays the workflow history and rebuilds these fields exactly as they were.

```
User message
    │
    ▼
Workflow stores message in state
    │
    ├─▶ call_llm Activity
    │
    ├─▶ wait for human approval signal
    │
    ├─▶ run tool Activity
    │
    └─▶ append final agent reply to conversation
```


In [ ]:
@dataclass
class AgentSessionInput:
    session_id: str
    messages: list[dict[str, str]] = field(default_factory=list)
    total_tokens: int = 0
    tools_used: list[str] = field(default_factory=list)
    turns_processed: int = 0
    max_turns_before_compaction: int = 50


@workflow.defn
class AgentWorkflow:
    def __init__(self):
        self._messages: list[dict[str, str]] = []
        self._status = "idle"
        self._pending_message: str | None = None
        self._approved = False
        self._rejected = False
        self._shutdown_requested = False
        self._total_tokens = 0
        self._tools_used: list[str] = []
        self._session_id = "unknown"
        self._turns_processed = 0
        self._max_turns_before_compaction = 50

    def _snapshot(self) -> AgentSessionInput:
        return AgentSessionInput(
            session_id=self._session_id,
            messages=[dict(message) for message in self._messages],
            total_tokens=self._total_tokens,
            tools_used=list(self._tools_used),
            turns_processed=self._turns_processed,
            max_turns_before_compaction=self._max_turns_before_compaction,
        )

    @workflow.signal
    def send_message(self, message: str) -> None:
        self._pending_message = message

    @workflow.signal
    def approve_action(self) -> None:
        self._approved = True
        self._rejected = False

    @workflow.signal
    def reject_action(self) -> None:
        self._rejected = True
        self._approved = False

    @workflow.signal
    def shutdown(self) -> None:
        self._shutdown_requested = True

    @workflow.query
    def get_status(self) -> str:
        return self._status

    @workflow.query
    def get_conversation(self) -> list[dict[str, str]]:
        return [dict(message) for message in self._messages]

    @workflow.query
    def get_stats(self) -> dict[str, object]:
        return {
            "total_tokens": self._total_tokens,
            "message_count": len(self._messages),
            "tools_used": list(self._tools_used),
            "status": self._status,
            "turns_processed": self._turns_processed,
        }

    @workflow.run
    async def run(self, session: AgentSessionInput) -> str:
        self._session_id = session.session_id
        self._messages = [dict(message) for message in session.messages]
        self._status = "waiting_for_input"
        self._total_tokens = session.total_tokens
        self._tools_used = list(session.tools_used)
        self._turns_processed = session.turns_processed
        self._max_turns_before_compaction = session.max_turns_before_compaction

        while not self._shutdown_requested:
            await workflow.wait_condition(
                lambda: self._pending_message is not None or self._shutdown_requested
            )

            if self._shutdown_requested:
                break

            user_message = self._pending_message or ""
            self._pending_message = None
            self._messages.append({"role": "user", "content": user_message})

            self._status = "thinking"
            first_pass = await workflow.execute_activity(
                call_llm,
                LlmRequest(prompt=user_message),
                start_to_close_timeout=timedelta(minutes=2),
                retry_policy=RetryPolicy(maximum_attempts=3),
            )
            self._total_tokens += first_pass.tokens_used

            planned_tools: list[tuple[str, str]] = []
            lowered = user_message.lower()
            if "search" in lowered:
                planned_tools.append(("search_web", user_message))
            if "code" in lowered or "python" in lowered or "calculate" in lowered:
                planned_tools.append(("execute_code", "print(6 * 7)"))

            tool_outputs: list[str] = []
            for tool_name, tool_input in planned_tools:
                if tool_name == "search_web":
                    self._status = "awaiting_approval"
                    try:
                        await workflow.wait_condition(
                            lambda: self._approved or self._rejected or self._shutdown_requested,
                            timeout=timedelta(minutes=5),
                        )
                    except asyncio.TimeoutError:
                        tool_outputs.append("Web search skipped because approval timed out.")
                        self._status = "thinking"
                        continue

                    if self._shutdown_requested:
                        break

                    if self._rejected:
                        tool_outputs.append("A human reviewer rejected the web search request.")
                    else:
                        self._status = "using_tool"
                        search_result = await workflow.execute_activity(
                            search_web,
                            tool_input,
                            start_to_close_timeout=timedelta(seconds=30),
                            retry_policy=RetryPolicy(maximum_attempts=3),
                        )
                        self._tools_used.append("search_web")
                        tool_outputs.append(search_result)

                    self._approved = False
                    self._rejected = False
                    self._status = "thinking"
                    continue

                self._status = "using_tool"
                code_result = await workflow.execute_activity(
                    execute_code,
                    tool_input,
                    start_to_close_timeout=timedelta(seconds=30),
                    retry_policy=RetryPolicy(maximum_attempts=2),
                )
                self._tools_used.append("execute_code")
                tool_outputs.append(code_result)
                self._status = "thinking"

            if self._shutdown_requested:
                break

            final_answer = first_pass.text
            if tool_outputs:
                synthesis = await workflow.execute_activity(
                    call_llm,
                    LlmRequest(
                        prompt=(
                            f"User message: {user_message}
"
                            f"Initial answer: {first_pass.text}
"
                            f"Tool outputs: {' | '.join(tool_outputs)}
"
                            "Write a final helpful answer for the user."
                        )
                    ),
                    start_to_close_timeout=timedelta(minutes=2),
                    retry_policy=RetryPolicy(maximum_attempts=3),
                )
                self._total_tokens += synthesis.tokens_used
                final_answer = synthesis.text + "

Tool outputs:
- " + "
- ".join(tool_outputs)

            self._messages.append({"role": "agent", "content": final_answer})
            self._turns_processed += 1
            self._status = "waiting_for_input"

            if self._turns_processed >= self._max_turns_before_compaction:
                self._status = "compacting"
                workflow.continue_as_new(args=[self._snapshot()])

        self._status = "completed"
        return (
            f"Agent session {self._session_id} completed after "
            f"{self._turns_processed} turns and {len(self._messages)} messages."
        )


print("✅ Defined AgentWorkflow with signals, queries, durable state, and Continue-As-New")


In [ ]:
async def demo_ai_agent():
    agent_input = AgentSessionInput(
        session_id="support-session-001",
        max_turns_before_compaction=10,
    )

    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[AgentWorkflow],
        activities=[call_llm, search_web, execute_code],
    ):
        workflow_id = f"ai-agent-{uuid.uuid4()}"
        handle = await client.start_workflow(
            AgentWorkflow.run,
            agent_input,
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )
        observer = client.get_workflow_handle(workflow_id)

        print(f"🚀 Started workflow: {workflow_id}")
        print("
📨 User asks for research...")
        await handle.signal(
            AgentWorkflow.send_message,
            "Search for Temporal Continue-As-New guidance and summarize what it means for AI agents.",
        )

        await asyncio.sleep(1.0)
        print("Status before approval:", await observer.query(AgentWorkflow.get_status))
        print("Conversation so far:", await observer.query(AgentWorkflow.get_conversation))

        print("
👩‍💼 Human supervisor approves the web search tool...")
        await handle.signal(AgentWorkflow.approve_action)

        await asyncio.sleep(1.5)
        print("Status after tool use:", await observer.query(AgentWorkflow.get_status))
        print("Stats:", await observer.query(AgentWorkflow.get_stats))

        print("
📨 User asks for a rollout plan...")
        await handle.signal(
            AgentWorkflow.send_message,
            "Plan a safe rollout for this durable AI agent system.",
        )

        await asyncio.sleep(1.0)
        conversation = await observer.query(AgentWorkflow.get_conversation)
        final_stats = await observer.query(AgentWorkflow.get_stats)
        print("
🧾 Conversation history:")
        for message in conversation:
            print(f"- {message['role']}: {message['content']}")

        print("
🛑 Shutting down the session...")
        await handle.signal(AgentWorkflow.shutdown)
        result = await handle.result()
        return {
            "workflow_id": workflow_id,
            "result": result,
            "stats": final_stats,
        }


demo_result = await demo_ai_agent()
print("
✅ Demo complete")
print(demo_result)


---
## 💥 Exercise: Why a worker crash does not lose the agent

Imagine the worker crashes right after the workflow stores the user message but before the tool call finishes.

```
Signal arrives ─▶ message stored in workflow state ─▶ tool Activity starts ─▶ X worker crashes
                       │
                       └──── Temporal already recorded the durable event history

New worker starts ─▶ workflow replays from history ─▶ missing work resumes safely
```

The important idea is that Temporal does **not** trust the worker's RAM. It trusts the event history stored by the Temporal service. Replay rebuilds the workflow state, so the agent can continue from the last durable step instead of starting over.


## 🛰️ Exercise: Query the agent from outside

A dashboard, API route, or human approval service can reconnect to the agent later as long as it knows the **workflow ID**.

Queries are perfect for:
- showing the latest status in a UI
- loading the current conversation into an admin console
- checking usage statistics before approving a risky action


In [ ]:
async def external_observer_example(workflow_id: str):
    handle = client.get_workflow_handle(workflow_id)
    return {
        "status": await handle.query(AgentWorkflow.get_status),
        "stats": await handle.query(AgentWorkflow.get_stats),
        "conversation": await handle.query(AgentWorkflow.get_conversation),
    }


print("✅ Defined external_observer_example(workflow_id)")
print("   In a real system this could live in a FastAPI route, admin dashboard, or approval service.")


---
## 🔁 Part 4: Continue-As-New for long-lived agent sessions

AI sessions can last for hours, days, or even forever. Every signal, every tool call, and every state transition adds more event history. If that history grows forever, replay gets slower and the workflow becomes harder to manage.

That is why the workflow above checks `max_turns_before_compaction` and eventually calls `workflow.continue_as_new(...)`.

```
Run 1: turns 1-50
        │  snapshot(messages, tokens, tools)
        └─▶ continue_as_new(snapshot)
                   │
                   └─▶ Run 2: turns 51-100
                              │
                              └─▶ ...same workflow ID, fresh run ID, small history
```

In this notebook we keep the threshold high so the demo stays easy to follow. In production you would compact based on turn count, token count, or Temporal history guidance.


---
## 🔌 MCP Integration Patterns

**MCP** (Model Context Protocol) is a standard way for AI systems to discover and call tools. Temporal fits naturally underneath it.

| MCP Concept | Temporal Mapping |
|---|---|
| Tool | Activity |
| Tool execution | `workflow.execute_activity()` |
| Tool approval | Signal + `workflow.wait_condition()` |
| Session | Workflow Execution |
| Memory | Workflow state |

A simple way to think about it is:
1. the model decides a tool should be used
2. the workflow records that decision durably
3. a signal can approve or reject the action
4. the activity performs the side effect
5. the workflow stores the result and decides what to do next

That gives you a tool-calling agent that is not just smart, but also **durable, observable, and controllable**.


## 🎓 What You Learned

- AI agents are a great fit for Temporal because they are long-running and failure-prone.
- LLM calls and tool calls should be Activities so they can retry safely.
- Workflow variables act like durable memory for the conversation and agent status.
- Signals let humans approve or reject risky tool calls.
- Queries let outside systems inspect status, conversation history, and usage stats.
- Continue-As-New keeps long-lived agent sessions healthy by compacting history.
- MCP-style tool use maps cleanly onto Temporal workflows and activities.
